In [1]:
import os
import hashlib
import random
import numpy as np
import pandas as pd

os.environ["PYTHONHASHSEED"] = "0"
random.seed(0)
np.random.seed(0)

SUBMISSIONS_PATH = "/kaggle/input"

DATA_DIR_CANDIDATES = [
    "/kaggle/input/plant-pathology-2020-fgvc7",
    "/kaggle/data/plant-pathology-2020-fgvc7",
    "/kaggle/input",
    "/kaggle/data",
]


def _find_file(filename, roots):
    for r in roots:
        p = os.path.join(r, filename)
        if os.path.exists(p):
            return p
    for r in roots:
        if os.path.exists(r):
            for dirpath, _, filenames in os.walk(r):
                if filename in filenames:
                    return os.path.join(dirpath, filename)
    return None


sample_path = _find_file("sample_submission.csv", DATA_DIR_CANDIDATES)
test_path = _find_file("test.csv", DATA_DIR_CANDIDATES)
train_path = _find_file("train.csv", DATA_DIR_CANDIDATES)

if sample_path is None or test_path is None:
    raise FileNotFoundError(
        f"Could not locate required files. sample_path={sample_path}, test_path={test_path}"
    )

print("Using sample_submission:", sample_path)
print("Using test.csv:", test_path)
print("Using train.csv:", train_path)



Using sample_submission: /kaggle/input/plant-pathology-2020-fgvc7/sample_submission.csv
Using test.csv: /kaggle/input/plant-pathology-2020-fgvc7/test.csv
Using train.csv: /kaggle/input/plant-pathology-2020-fgvc7/train.csv


In [2]:
submissions_all = []
for dirname, _, filenames in os.walk(SUBMISSIONS_PATH):
    for filename in filenames:
        if not filename.lower().endswith(".csv"):
            continue
        full = os.path.join(dirname, filename)
        base = os.path.basename(full).lower()
        if base in {"sample_submission.csv", "train.csv", "test.csv"}:
            continue
        submissions_all.append(full)

submissions_all.sort()
print("Found candidate external submissions:", len(submissions_all))
print(submissions_all[:20])




Found candidate external submissions: 0
[]


In [3]:
def ensemble(submissions_all, sub_idx, weights=None, test_csv_path=None):
    """
    Weighted sum ensemble of provided submission files.

    Align each external submission by image_id to the test.csv order to avoid row-order mismatches.
    Core ensembling logic (weighted sum) is unchanged.
    """
    if weights is None:
        weights = [1.0] * len(sub_idx)
    if len(sub_idx) != len(weights):
        raise ValueError(
            f"sub_idx and weights must have same length. Got {len(sub_idx)} vs {len(weights)}"
        )

    needed = ["healthy", "multiple_diseases", "rust", "scab"]

    test_ids = None
    if test_csv_path is not None:
        test_df = pd.read_csv(test_csv_path)
        test_ids = test_df["image_id"].astype(str).values

    submission_with_weight = []
    for i in range(len(sub_idx)):
        idx = sub_idx[i]
        if idx < 0 or idx >= len(submissions_all):
            raise IndexError(
                f"Requested submission index {idx} but only {len(submissions_all)} files available."
            )
        path = submissions_all[idx]
        w = float(weights[i])
        print(f"I'm taking submission {path} with weight {w}")

        df = pd.read_csv(path)
        missing = [c for c in (["image_id"] + needed) if c not in df.columns]
        if missing:
            raise ValueError(f"Submission file {path} is missing columns: {missing}")

        if test_ids is not None:
            df = df.copy()
            df["image_id"] = df["image_id"].astype(str)
            df = df.set_index("image_id").reindex(test_ids).reset_index()
            if df[needed].isna().any().any():
                raise ValueError(
                    f"Submission file {path} has missing predictions after reindexing to test.csv order."
                )

        arr = df.loc[:, needed].to_numpy(dtype="float64")
        submission_with_weight.append(arr * w)

    submission_sum = submission_with_weight[0]
    for arr in submission_with_weight[1:]:
        submission_sum = submission_sum + arr
    return submission_sum




In [4]:
def make_submission_file(
    submission_avg, sample_submission_path, test_csv_path, out_path="submission.csv"
):
    """
    Ensure correct columns/order and that image_id matches test.csv; clip to [0,1].
    """
    sample_df = pd.read_csv(sample_submission_path)
    test_df = pd.read_csv(test_csv_path)

    out = sample_df.copy()
    out = out.set_index("image_id")
    out = out.reindex(test_df["image_id"].values)
    out = out.reset_index()

    needed = ["healthy", "multiple_diseases", "rust", "scab"]
    if submission_avg is None:
        raise ValueError("submission_avg cannot be None")

    pred = pd.DataFrame(submission_avg, columns=needed)
    pred = pred.clip(0.0, 1.0)

    if len(pred) != len(out):
        raise ValueError(
            f"Prediction rows ({len(pred)}) do not match test rows ({len(out)})"
        )

    out[needed] = pred[needed].values
    out.to_csv(out_path, index=False)
    print(f"Wrote {out_path} with shape {out.shape} and columns {list(out.columns)}")




In [5]:
def _id_to_u01_salt(image_id: str, salt: str) -> float:
    """Deterministic pseudo-random in [0,1) from image_id + salt (stable across runs)."""
    h = hashlib.md5((salt + "::" + image_id).encode("utf-8")).hexdigest()
    return int(h[:8], 16) / float(16**8)


def _rank_to_uniform(x):
    """Map scores to (0,1) via ranks so each class has a full spread; AUC depends on ordering."""
    s = pd.Series(x)
    r = s.rank(method="average").to_numpy(dtype="float64")
    n = float(len(r))
    return r / (n + 1.0)


def _find_images_dir(roots):
    candidates = []
    for r in roots:
        p = os.path.join(r, "images")
        if os.path.isdir(p):
            candidates.append(p)
    if candidates:
        return candidates[0]

    for r in roots:
        if os.path.exists(r):
            for dirpath, dirnames, _ in os.walk(r):
                if "images" in dirnames:
                    return os.path.join(dirpath, "images")
    return None


def _resolve_image_path(images_dir, image_id):
    p1 = os.path.join(images_dir, f"{image_id}.jpg")
    if os.path.exists(p1):
        return p1
    p2 = os.path.join(images_dir, f"{image_id}.JPG")
    if os.path.exists(p2):
        return p2
    return None


def _image_features(image_path, resize_hw=(192, 192)):
    """Uses lightweight deterministic image features (same feature definitions; resize is a tuned hyperparameter)."""
    try:
        from PIL import Image
    except Exception:
        return None

    try:
        with Image.open(image_path) as im:
            im = im.convert("RGB")
            im = im.resize(resize_hw)
            arr = np.asarray(im, dtype="float32") / 255.0  # (H,W,3)
    except Exception:
        return None

    try:
        r = arr[:, :, 0]
        g = arr[:, :, 1]
        b = arr[:, :, 2]

        r_mean = float(r.mean())
        g_mean = float(g.mean())
        b_mean = float(b.mean())
        r_std = float(r.std())
        g_std = float(g.std())
        b_std = float(b.std())

        eps = 1e-6
        sum_rgb = r + g + b + eps
        r_frac = float((r / sum_rgb).mean())
        g_frac = float((g / sum_rgb).mean())
        b_frac = float((b / sum_rgb).mean())

        exg = float((2.0 * g - r - b).mean())
        exr = float((1.4 * r - g).mean())
        rg = float(((r + eps) / (g + eps)).mean())
        gb = float(((g + eps) / (b + eps)).mean())
        rb = float(((r + eps) / (b + eps)).mean())

        gray = 0.2989 * r + 0.5870 * g + 0.1140 * b
        dx = gray[:, 1:] - gray[:, :-1]
        dy = gray[1:, :] - gray[:-1, :]
        edge = float((np.abs(dx).mean() + np.abs(dy).mean()) / 2.0)

        brightness = float(gray.mean())
        contrast = float(gray.std())

        from PIL import Image

        arr_u8 = (arr * 255.0).clip(0, 255).astype("uint8")
        hsv = (
            np.asarray(
                Image.fromarray(arr_u8, mode="RGB").convert("HSV"), dtype="float32"
            )
            / 255.0
        )
        h = hsv[:, :, 0]  # [0,1) circular
        s = hsv[:, :, 1]
        v = hsv[:, :, 2]
        s_mean = float(s.mean())
        v_mean = float(v.mean())
        s_std = float(s.std())
        v_std = float(v.std())

        center = gray[1:-1, 1:-1]
        lap_valid = (
            -4.0 * center
            + gray[:-2, 1:-1]
            + gray[2:, 1:-1]
            + gray[1:-1, :-2]
            + gray[1:-1, 2:]
        )
        lap_energy = float(np.mean(np.abs(lap_valid)))

        yellow_frac = float(((h > 0.08) & (h < 0.22) & (s > 0.25) & (v > 0.25)).mean())
        green_def_frac = float(((g < 0.35) & (r > 0.35) & (s > 0.20)).mean())

        orange_frac = float(((h >= 0.03) & (h < 0.10) & (s > 0.25) & (v > 0.20)).mean())
        brown_frac = float(((h >= 0.03) & (h < 0.14) & (s > 0.20) & (v < 0.55)).mean())
        green_frac_hsv = float(
            ((h >= 0.22) & (h < 0.45) & (s > 0.15) & (v > 0.15)).mean()
        )

        dark_spot_frac = float(((v < 0.35) & (s > 0.15)).mean())

        return (
            r_mean,
            g_mean,
            b_mean,
            r_std,
            g_std,
            b_std,
            r_frac,
            g_frac,
            b_frac,
            exg,
            exr,
            rg,
            gb,
            rb,
            edge,
            brightness,
            contrast,
            s_mean,
            v_mean,
            s_std,
            v_std,
            lap_energy,
            yellow_frac,
            green_def_frac,
            orange_frac,
            brown_frac,
            green_frac_hsv,
            dark_spot_frac,
        )
    except Exception:
        return None


def fallback_predictions_from_priors_and_images(
    train_csv_path, test_csv_path, data_roots
):
    """
    Same pipeline as before:
    features -> weighted ridge per class -> OOF affine calibration -> rank-to-uniform -> prior mix -> submission.

    Change to improve score toward target (minimal, metric-aligned):
    - Tune (resize, sample-weight power, ridge lambda, alpha) by OOF mean ROC AUC using the same post-processing.
      This does not change the model family or approach; it just picks better fixed hyperparameters.
    """
    needed = ["healthy", "multiple_diseases", "rust", "scab"]

    test_df = pd.read_csv(test_csv_path)
    image_ids_test = test_df["image_id"].astype(str).tolist()

    if train_csv_path is not None and os.path.exists(train_csv_path):
        train_df = pd.read_csv(train_csv_path)
        priors = train_df[needed].mean().to_dict()
        image_ids_train = train_df["image_id"].astype(str).tolist()
        y_train = train_df[needed].to_numpy(dtype="float64")
    else:
        priors = {c: 0.25 for c in needed}
        image_ids_train = []
        y_train = None

    images_dir = _find_images_dir(data_roots)
    print("Using images_dir:", images_dir)

    try:
        from sklearn.metrics import roc_auc_score
    except Exception:
        roc_auc_score = None

    # Precompute deterministic folds once (keeps training approach identical; avoids redoing work per grid point)
    if y_train is not None and len(image_ids_train) > 0:
        n = len(image_ids_train)
        dom = np.argmax(y_train, axis=1).astype(int)

        class_bins = {k: [] for k in range(y_train.shape[1])}
        for i, (cid, imgid) in enumerate(zip(dom.tolist(), image_ids_train)):
            class_bins[cid].append((i, _id_to_u01_salt(str(imgid), "fold_seed_v1")))
        for k in class_bins:
            class_bins[k].sort(key=lambda t: t[1])

        fold = np.zeros((n,), dtype=int)
        K = 5
        for k in class_bins:
            idxs = [t[0] for t in class_bins[k]]
            for j, ii in enumerate(idxs):
                fold[ii] = j % K
    else:
        fold = None
        K = 5

    def _affine_fit(x, y):
        x_mean = float(x.mean())
        y_mean = float(y.mean())
        x_var = float(((x - x_mean) ** 2).mean())
        if x_var < 1e-12:
            a = 1.0
        else:
            a = float(((x - x_mean) * (y - y_mean)).mean() / x_var)
        b0 = y_mean - a * x_mean
        return a, b0

    def _rank_to_uniform_using_reference(x, ref):
        ref_sorted = (
            pd.Series(ref).sort_values(kind="mergesort").to_numpy(dtype="float64")
        )
        nref = float(len(ref_sorted))
        x = pd.Series(x).to_numpy(dtype="float64")
        left = np.searchsorted(ref_sorted, x, side="left")
        right = np.searchsorted(ref_sorted, x, side="right")
        avg = (left + right) / 2.0
        return (avg + 1.0) / (nref + 2.0)

    def _fit_ridge_weighted(X, y, reg, sqrt_w_local):
        Xw = X * sqrt_w_local
        yw = y * sqrt_w_local[:, 0]
        A = Xw.T @ Xw + reg
        bvec = Xw.T @ yw
        return np.linalg.solve(A, bvec)

    def _featurize_ids(image_ids, resize_hw, feat_cache):
        X = np.zeros((len(image_ids), 28 + 3), dtype="float64")
        for i, image_id in enumerate(image_ids):
            u0 = _id_to_u01_salt(image_id, "pp2020_s0")
            u1 = _id_to_u01_salt(image_id, "pp2020_s1")
            u2 = _id_to_u01_salt(image_id, "pp2020_s2")
            z0, z1, z2 = (u0 - 0.5, u1 - 0.5, u2 - 0.5)

            feats28 = (0.0,) * 28
            if images_dir is not None:
                key = (image_id, int(resize_hw[0]), int(resize_hw[1]))
                if key in feat_cache:
                    feats28 = feat_cache[key]
                else:
                    img_path = _resolve_image_path(images_dir, image_id)
                    if img_path is not None:
                        f = _image_features(img_path, resize_hw=resize_hw)
                        if f is not None:
                            feats28 = f
                    feat_cache[key] = feats28

            X[i, :28] = np.asarray(feats28, dtype="float64")
            X[i, 28:] = np.asarray([z0, z1, z2], dtype="float64")
        return X

    # If no train labels, keep a simple deterministic fallback (unchanged semantics)
    if y_train is None or len(image_ids_train) == 0:
        feat_cache = {}
        X_test = _featurize_ids(
            image_ids_test, resize_hw=(192, 192), feat_cache=feat_cache
        )
        raw = {}
        for j, c in enumerate(needed):
            raw[c] = X_test[:, j % X_test.shape[1]]
        preds = []
        for c in needed:
            u = _rank_to_uniform(raw[c])
            p = 0.92 * u + 0.08 * float(priors[c])
            preds.append(p)
        pred = pd.DataFrame({c: preds[i] for i, c in enumerate(needed)}).clip(0.0, 1.0)
        return pred.to_numpy(dtype="float64")

    # Small fixed grids to keep runtime bounded while improving OOF selection (metric-aligned)
    resize_grid = [(160, 160), (192, 192), (224, 224)]
    weight_power_grid = [0.6, 1.0, 1.4]
    lam_grid = [0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]
    alpha_grid = [0.88, 0.90, 0.92, 0.94, 0.96]

    best = {
        "score": -1e18,
        "resize": (192, 192),
        "weight_power": 1.0,
        "lam": lam_grid[0],
        "alpha": 0.92,
    }

    # Feature cache across grid points (keyed by (image_id, resize))
    feat_cache = {}

    # We keep the exact same post-processing used for submission inside the OOF scoring.
    for resize_hw in resize_grid:
        X_train = _featurize_ids(
            image_ids_train, resize_hw=resize_hw, feat_cache=feat_cache
        )
        X_test = _featurize_ids(
            image_ids_test, resize_hw=resize_hw, feat_cache=feat_cache
        )

        mu = X_train.mean(axis=0)
        sigma = X_train.std(axis=0)
        sigma[sigma < 1e-6] = 1.0
        Xtr = (X_train - mu) / sigma
        Xte = (X_test - mu) / sigma

        Xtr_i = np.concatenate(
            [np.ones((Xtr.shape[0], 1), dtype="float64"), Xtr], axis=1
        )
        Xte_i = np.concatenate(
            [np.ones((Xte.shape[0], 1), dtype="float64"), Xte], axis=1
        )

        n = Xtr_i.shape[0]
        p_dim = Xtr_i.shape[1]
        I = np.eye(p_dim, dtype="float64")
        I[0, 0] = 0.0  # do not regularize intercept

        dom = np.argmax(y_train, axis=1).astype(int)
        counts = np.bincount(dom, minlength=y_train.shape[1]).astype("float64")
        counts[counts < 1.0] = 1.0
        inv = 1.0 / counts

        for weight_power in weight_power_grid:
            # Change (score improvement): tune sample-weight strength; same weighting concept, just a better fixed exponent.
            w = (inv[dom]) ** float(weight_power)
            w = w / float(w.mean())
            sqrt_w = np.sqrt(w).reshape(-1, 1)

            for lam in lam_grid:
                reg = lam * I
                raw_oof = np.zeros((n, y_train.shape[1]), dtype="float64")

                for f in range(K):
                    tr_mask = fold != f
                    va_mask = ~tr_mask

                    Xtr_f = Xtr_i[tr_mask]
                    ytr_f = y_train[tr_mask]
                    Xva_f = Xtr_i[va_mask]
                    sqrt_w_f = sqrt_w[tr_mask]

                    W_f = np.zeros((p_dim, y_train.shape[1]), dtype="float64")
                    for kk in range(y_train.shape[1]):
                        W_f[:, kk] = _fit_ridge_weighted(
                            Xtr_f, ytr_f[:, kk], reg, sqrt_w_f
                        )

                    raw_oof[va_mask] = Xva_f @ W_f

                # OOF affine calibration (unchanged)
                raw_oof_cal = np.zeros_like(raw_oof)
                for f in range(K):
                    tr_mask = fold != f
                    va_mask = ~tr_mask
                    for kk in range(y_train.shape[1]):
                        a, b0 = _affine_fit(raw_oof[tr_mask, kk], y_train[tr_mask, kk])
                        raw_oof_cal[va_mask, kk] = raw_oof[va_mask, kk] * a + b0

                # Fold-local rank-to-uniform (unchanged)
                oof_u = np.zeros_like(raw_oof_cal)
                for f in range(K):
                    tr_mask = fold != f
                    va_mask = ~tr_mask
                    for kk in range(y_train.shape[1]):
                        oof_u[va_mask, kk] = _rank_to_uniform_using_reference(
                            raw_oof_cal[va_mask, kk], raw_oof_cal[tr_mask, kk]
                        )

                # Tune alpha (prior mix) on the same OOF output (same semantics)
                pri = np.array(
                    [float(priors[c]) for c in needed], dtype="float64"
                ).reshape(1, -1)
                for alpha in alpha_grid:
                    oof_p = float(alpha) * oof_u + (1.0 - float(alpha)) * pri

                    if roc_auc_score is not None:
                        aucs = []
                        for kk in range(y_train.shape[1]):
                            try:
                                aucs.append(
                                    float(roc_auc_score(y_train[:, kk], oof_p[:, kk]))
                                )
                            except Exception:
                                pass
                        score = float(sum(aucs) / len(aucs)) if len(aucs) else -1e18
                    else:
                        score = float(np.mean(np.var(oof_p, axis=0)))

                    if score > best["score"]:
                        best.update(
                            {
                                "score": score,
                                "resize": resize_hw,
                                "weight_power": float(weight_power),
                                "lam": float(lam),
                                "alpha": float(alpha),
                            }
                        )

        print(
            f"Completed resize={resize_hw}. Current best OOF mean AUC={best['score']:.6f} "
            f"(resize={best['resize']}, weight_power={best['weight_power']}, lam={best['lam']}, alpha={best['alpha']})"
        )

    print("Selected hyperparams:", best)

    # Refit with selected hyperparams on all train, then predict test (same inference steps as before)
    resize_hw = best["resize"]
    weight_power = best["weight_power"]
    lam = best["lam"]
    alpha = best["alpha"]

    X_train = _featurize_ids(
        image_ids_train, resize_hw=resize_hw, feat_cache=feat_cache
    )
    X_test = _featurize_ids(image_ids_test, resize_hw=resize_hw, feat_cache=feat_cache)

    mu = X_train.mean(axis=0)
    sigma = X_train.std(axis=0)
    sigma[sigma < 1e-6] = 1.0
    Xtr = (X_train - mu) / sigma
    Xte = (X_test - mu) / sigma

    Xtr_i = np.concatenate([np.ones((Xtr.shape[0], 1), dtype="float64"), Xtr], axis=1)
    Xte_i = np.concatenate([np.ones((Xte.shape[0], 1), dtype="float64"), Xte], axis=1)

    n = Xtr_i.shape[0]
    p_dim = Xtr_i.shape[1]
    I = np.eye(p_dim, dtype="float64")
    I[0, 0] = 0.0

    dom = np.argmax(y_train, axis=1).astype(int)
    counts = np.bincount(dom, minlength=y_train.shape[1]).astype("float64")
    counts[counts < 1.0] = 1.0
    inv = 1.0 / counts
    w = (inv[dom]) ** float(weight_power)
    w = w / float(w.mean())
    sqrt_w = np.sqrt(w).reshape(-1, 1)

    reg = float(lam) * I
    W = np.zeros((p_dim, y_train.shape[1]), dtype="float64")
    for k in range(y_train.shape[1]):
        W[:, k] = _fit_ridge_weighted(Xtr_i, y_train[:, k], reg, sqrt_w)

    raw_test = Xte_i @ W

    # OOF raw for calibration on full train (unchanged)
    raw_oof = np.zeros((n, y_train.shape[1]), dtype="float64")
    for f in range(K):
        tr_mask = fold != f
        va_mask = ~tr_mask
        Xtr_f = Xtr_i[tr_mask]
        ytr_f = y_train[tr_mask]
        Xva_f = Xtr_i[va_mask]
        sqrt_w_f = sqrt_w[tr_mask]

        W_f = np.zeros((p_dim, y_train.shape[1]), dtype="float64")
        for kk in range(y_train.shape[1]):
            W_f[:, kk] = _fit_ridge_weighted(Xtr_f, ytr_f[:, kk], reg, sqrt_w_f)
        raw_oof[va_mask] = Xva_f @ W_f

    cal_a = np.ones((y_train.shape[1],), dtype="float64")
    cal_b = np.zeros((y_train.shape[1],), dtype="float64")
    for k in range(y_train.shape[1]):
        a, b0 = _affine_fit(raw_oof[:, k], y_train[:, k])
        cal_a[k] = a
        cal_b[k] = b0

    raw_test_cal = raw_test * cal_a.reshape(1, -1) + cal_b.reshape(1, -1)

    preds = []
    for k, c in enumerate(needed):
        u = _rank_to_uniform(raw_test_cal[:, k])
        p = float(alpha) * u + (1.0 - float(alpha)) * float(priors[c])
        preds.append(p)

    pred = pd.DataFrame({c: preds[i] for i, c in enumerate(needed)}).clip(0.0, 1.0)
    return pred.to_numpy(dtype="float64")




In [6]:
if len(submissions_all) >= 3:
    submission_avg = ensemble(
        submissions_all, [0, 1, 2], [0.04, 0.2, 0.04], test_csv_path=test_path
    )
else:
    print(
        "No (or insufficient) external submission files found to ensemble; using image-feature + rank-based fallback to improve ROC AUC signal."
    )
    submission_avg = fallback_predictions_from_priors_and_images(
        train_path, test_path, DATA_DIR_CANDIDATES
    )

make_submission_file(submission_avg, sample_path, test_path, out_path="submission.csv")

No (or insufficient) external submission files found to ensemble; using image-feature + rank-based fallback to improve ROC AUC signal.
Using images_dir: /kaggle/input/plant-pathology-2020-fgvc7/images


/tmp/ipykernel_11/3646380030.py:94: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(arr_u8, mode="RGB").convert("HSV"), dtype="float32"


Completed resize=(160, 160). Current best OOF mean AUC=0.692584 (resize=(160, 160), weight_power=0.6, lam=0.1, alpha=0.88)


/tmp/ipykernel_11/3646380030.py:94: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(arr_u8, mode="RGB").convert("HSV"), dtype="float32"


Completed resize=(192, 192). Current best OOF mean AUC=0.692770 (resize=(192, 192), weight_power=0.6, lam=0.003, alpha=0.88)


/tmp/ipykernel_11/3646380030.py:94: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(arr_u8, mode="RGB").convert("HSV"), dtype="float32"


Completed resize=(224, 224). Current best OOF mean AUC=0.692770 (resize=(192, 192), weight_power=0.6, lam=0.003, alpha=0.88)
Selected hyperparams: {'score': 0.692770137008905, 'resize': (192, 192), 'weight_power': 0.6, 'lam': 0.003, 'alpha': 0.88}
Wrote submission.csv with shape (183, 5) and columns ['image_id', 'healthy', 'multiple_diseases', 'rust', 'scab']
